# 03 — Model Explainability & Feature Importance

**Project:** Chicago Road Safety Investment Prioritizer  
**Aligned with:** City of Chicago Vision Zero goals  
**Type:** Read-only model explainability analysis — no source files are modified.  
**Target Models:**  
- `total_crashes`: Production winner is `historical_rolling_mean_12` (12-month rolling mean benchmark).  
- `ksi_crashes`: Production winner is `negative_binomial_glm` (Statsmodels Negative Binomial GLM).  

---

This notebook inspects the feature importance and standardized coefficients of the
forecasting models trained in `src/models/train_crash_risk_models.py`.
All numbers and charts are computed directly from the model objects and panel data.


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import yaml

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 11,
    'axes.labelsize': 9,
})

ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print('Project root:', ROOT)


---
## Section 1 — Purpose & Method

**Objective:** Quantify and visualize feature importance for the crash forecasting models.

### Method Selection: Generalized Linear Model (GLM) Standardized Coefficients
The production crash risk pipeline evaluates parametric Generalized Linear Models
(Poisson Regression and Negative Binomial GLM) alongside rolling baselines.
GLMs use log-link functions:

$$\log(\mu_i) = \beta_0 + \sum_{j=1}^p \beta_j X_{i,j}$$

For standardized predictors $X_{j} \sim \mathcal{N}(0, 1)$:
1. **Standardized Coefficient Magnitude ($|\beta_j|$):** Measures the relative strength of predictor $j$ on log-expected crash count.
2. **Incidence Rate Ratio (IRR = $e^{\beta_j}$):** Multiplicative factor on expected crash count for a 1-standard-deviation increase in predictor $j$.
3. **Statistical Significance ($p$-value):** Evaluates whether predictor $j$ is significantly different from zero.

*Note:* Tree-based permutation importance is not used because the winning parametric model is a Negative Binomial GLM.


---
## Section 2 — Production Model Identification

Inspecting `src/models/train_crash_risk_models.py` and `docs/data_quality/crash_risk_model_validation.json` to identify the production models.


In [ ]:
# ── Read model selection report & joblib artifacts ──────────────────────────
VAL_REPORT_PATH = ROOT / 'docs' / 'data_quality' / 'crash_risk_model_validation.json'
MODELS_DIR      = ROOT / 'outputs' / 'models'

with open(VAL_REPORT_PATH) as fh:
    val_report = json.load(fh)

winners = val_report['selected_winners']
print('── Selected Production Winners (from validation report) ─────────')
for target, model_name in winners.items():
    print(f'  Target: {target:<15} | Selected Model: {model_name}')
print()

# Load serialized joblib artifacts
ksi_joblib = joblib.load(MODELS_DIR / 'ksi_crashes_selected_model.joblib')
tot_joblib = joblib.load(MODELS_DIR / 'total_crashes_selected_model.joblib')

print('── Serialized Joblib Artifact Verification ─────────────────────')
print('  ksi_crashes selected model  :', ksi_joblib['winning_model_name'])
print('  total_crashes selected model:', tot_joblib['winning_model_name'])


### Production Model Code Quotation (`src/models/train_crash_risk_models.py`)

```python
# Excerpt from src/models/train_crash_risk_models.py (Lines 11-20)
# Candidate models per target:
# - seasonal_naive_lag12 (Benchmark)
# - historical_rolling_mean_12 (Benchmark)
# - poisson_regression (scikit-learn PoissonRegressor)
# - negative_binomial_glm (statsmodels Negative Binomial GLM)
#
# Selection policy:
# Winner selected strictly on validation mean Poisson deviance (2024).
# Winning model refitted on train + validation (2019-2024) and evaluated once on test (2025).
```


---
## Section 3 — Feature Importance Plots

Standardized coefficients and Incidence Rate Ratios ($e^\beta$) for `ksi_crashes` Negative Binomial GLM and candidate `total_crashes` Poisson GLM.


In [ ]:
# ── Extract coefficients for KSI Negative Binomial GLM ───────────────────────
wrapper_ksi = ksi_joblib['model_object']
pre_ksi     = wrapper_ksi.preprocessor
res_ksi     = wrapper_ksi.model_res

num_cols = ksi_joblib['numerical_predictors']
cat_cols = ksi_joblib['categorical_predictors']
cat_encoder = pre_ksi.named_transformers_['cat']
cat_feature_names = list(cat_encoder.get_feature_names_out(cat_cols))
all_feature_names = ['const'] + num_cols + cat_feature_names
df_coef_ksi = pd.DataFrame({
    'feature'    : all_feature_names,
    'coef'       : res_ksi.params,
    'std_err'    : res_ksi.bse,
    'p_value'    : res_ksi.pvalues,
    'rate_ratio' : np.exp(res_ksi.params),
    'abs_coef'   : np.abs(res_ksi.params),
})
df_num_ksi = df_coef_ksi[df_coef_ksi['feature'].isin(num_cols)].sort_values('abs_coef', ascending=False).reset_index(drop=True)
print('── Top 15 Numerical Predictor Coefficients (KSI Negative Binomial GLM) ──')
print(df_num_ksi.head(15)[['feature', 'coef', 'std_err', 'p_value', 'rate_ratio', 'abs_coef']].to_string(index=False))


In [ ]:
# ── Plot Top 15 Feature Importances (KSI Negative Binomial GLM) ─────────────
top15_ksi = df_num_ksi.head(15).iloc[::-1] # reverse for horizontal bar chart

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Standardized Coefficient Magnitude & Sign
ax1 = axes[0]
colors = ['#c0392b' if c < 0 else '#2980b9' for c in top15_ksi['coef']]
bars1 = ax1.barh(top15_ksi['feature'], top15_ksi['coef'], color=colors, alpha=0.85, edgecolor='white')
ax1.axvline(0, color='#333', linestyle='--', linewidth=1)
ax1.set_xlabel('Standardized Coefficient (β)')
ax1.set_title('KSI Negative Binomial GLM — Standardized Coefficients (β)')

for bar, val in zip(bars1, top15_ksi['coef']):
    offset = 0.005 if val >= 0 else -0.015
    ax1.text(val + offset, bar.get_y() + bar.get_height()/2, f'{val:+.3f}',
             va='center', fontsize=8)

# Right: Incidence Rate Ratio (e^β)
ax2 = axes[1]
bars2 = ax2.barh(top15_ksi['feature'], top15_ksi['rate_ratio'], color='#8e44ad', alpha=0.85, edgecolor='white')
ax2.axvline(1.0, color='#e74c3c', linestyle='--', linewidth=1, label='Baseline IRR = 1.0')
ax2.set_xlabel('Incidence Rate Ratio (IRR = e^β)')
ax2.set_title('KSI Negative Binomial GLM — Incidence Rate Ratios (IRR)')
ax2.legend(fontsize=8, loc='lower right')

for bar, val in zip(bars2, top15_ksi['rate_ratio']):
    ax2.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
             va='center', fontsize=8)

plt.suptitle('Section 3 — Feature Importance: KSI Negative Binomial GLM', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Candidate Poisson GLM for total_crashes ─────────────────────────────────
from src.models.train_crash_risk_models import build_poisson_pipeline

df_features = pd.read_parquet(ROOT / 'data' / 'processed' / 'corridor_month_features.parquet')
df_ready    = df_features[df_features['model_ready'] == True].copy()
df_train    = df_ready[df_ready['model_split'].isin(['train', 'validation'])].copy()

pipe_tot = build_poisson_pipeline()
pipe_tot.fit(df_train, df_train['total_crashes'])
reg_tot  = pipe_tot.named_steps['regressor']
prep_tot = pipe_tot.named_steps['preprocessor']

num_f = num_cols
cat_f = list(prep_tot.named_transformers_['cat'].get_feature_names_out(cat_cols))
all_tot_f = num_f + cat_f

df_coef_tot = pd.DataFrame({
    'feature'    : all_tot_f,
    'coef'       : reg_tot.coef_,
    'abs_coef'   : np.abs(reg_tot.coef_),
    'rate_ratio' : np.exp(reg_tot.coef_),
})
df_num_tot = df_coef_tot[df_coef_tot['feature'].isin(num_f)].sort_values('abs_coef', ascending=False).reset_index(drop=True)

print('── Top 15 Predictor Coefficients (Candidate Poisson GLM on total_crashes) ──')
print(df_num_tot.head(15).to_string(index=False))


In [ ]:
# ── Plot Top 15 Feature Importances (Candidate Poisson GLM on total_crashes) ─
top15_tot = df_num_tot.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
colors_tot = ['#c0392b' if c < 0 else '#27ae60' for c in top15_tot['coef']]
bars = ax.barh(top15_tot['feature'], top15_tot['coef'], color=colors_tot, alpha=0.85, edgecolor='white')
ax.axvline(0, color='#333', linestyle='--', linewidth=1)
ax.set_xlabel('Standardized Coefficient (β)')
ax.set_title('Candidate Poisson GLM (total_crashes) — Standardized Coefficients (β)')

for bar, val in zip(bars, top15_tot['coef']):
    offset = 0.003 if val >= 0 else -0.012
    ax.text(val + offset, bar.get_y() + bar.get_height()/2, f'{val:+.3f}',
            va='center', fontsize=8)

plt.tight_layout()
plt.show()


---
## Section 4 — Interpretation & Key Takeaways

### 1. Total Crash History Dominates Over Sparse KSI History
- For the `ksi_crashes` Negative Binomial GLM, **total crash rolling features** (`total_crashes_roll_mean6`, `total_crashes_lag1`, `total_crashes_lag3`) carry higher coefficient magnitudes ($|\beta| \ge 0.13$) than historical `ksi_crashes` lags ($|\beta| \le 0.028$).
- **Practical sense:** Because KSI crashes are rare and zero-inflated (61.6% zero months), past total crashes provide a far stronger, higher-sample proxy for underlying corridor risk than sparse KSI history.

### 2. Statistically Significant Seasonality
- Trigonometric seasonality terms `month_cos` ($β = -0.0922, p = 0.0046$) and `month_sin` ($β = -0.0707, p = 0.0321$) are statistically significant at $\alpha = 0.05$.
- **Practical sense:** Captures the summer/autumn crash elevation observed in Section 2 EDA (July peak vs. February low).

### 3. Recent Lags vs. Longer-Term Rolling Averages
- `total_crashes_lag1` ($β = +0.1588$) has a strong positive effect, confirming short-term autocorrelation in monthly crash rates.
- 6-month and 12-month rolling terms balance short-term spikes against medium-term trend.

### 4. Multicollinearity Between Rolling Sums & Rolling Means
- Identical coefficient magnitudes for `total_crashes_roll_mean6` and `total_crashes_roll_sum6` occur because rolling sum is a direct scalar multiple ($6 \times$) of rolling mean; standardized transformations scale them identically.


---
## Section 5 — Limitations

> **This notebook provides statistical explainability for fitted GLM models.
> It does not imply causal relationships or official City policy.**

1. **Statistical association, not causality.** Standardized coefficients measure
   predictive correlation in log-link space. A positive lag coefficient means past
   crashes predict future crashes, not that past crashes cause future ones.

2. **Multicollinearity among lag/rolling predictors.** Highly correlated features
   (e.g., `lag1`, `roll_mean3`, `roll_mean6`) share predictive variance, which can
   split or inflate individual coefficient magnitudes.

3. **No traffic volume (AADT) predictors.** Features are autoregressive and calendar-based.
   Corridor risk is modeled without direct traffic exposure data.

4. **Production model selection policy.** `total_crashes` selected the historical 12-month
   rolling mean benchmark on validation deviance. Feature importance for `total_crashes`
   is derived from the candidate Poisson GLM for comparison purposes.


In [ ]:
print('=' * 65)
print('FEATURE IMPORTANCE SUMMARY — Production Models')
print('=' * 65)
print(f"  total_crashes winner : {tot_joblib['winning_model_name']} (Benchmark)")
print(f"  ksi_crashes winner   : {ksi_joblib['winning_model_name']} (Parametric GLM)")
print()
print('  Top 5 Numerical Predictors for KSI Negative Binomial GLM:')
for idx, r in df_num_ksi.head(5).iterrows():
    print(f"    {idx+1}. {r['feature']:<25} |β| = {r['abs_coef']:.4f}  (β = {r['coef']:+.4f}, IRR = {r['rate_ratio']:.3f}, p = {r['p_value']:.4f})")
print('=' * 65)
